In [17]:
import sys
sys.path.insert(0,'../g3algo/')
from g3groupfinder import giantmodel, decayexp, sigmarange
import foftools as fof
import iterativecombination as ic
from smoothedbootstrap import smoothedbootstrap as sbs
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator
from matplotlib import rcParams
from scipy.optimize import curve_fit
from center_binned_stats import center_binned_stats
rcParams['axes.labelsize'] = 9
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['legend.fontsize'] = 9
rcParams['font.family'] = 'sans-serif'
rcParams['grid.color'] = 'k'
rcParams['grid.linewidth'] = 0.2
my_locator = MaxNLocator(6)
singlecolsize = (3.3522420091324205, 2.0717995001590714)
doublecolsize = (7.100005949910059, 4.3880449973709)
HUBBLE_CONST = 70.

# Boundaries of Giant-only FoF groups

In [3]:
ecodata = pd.read_csv("ECOdata_G3catalog_luminosity.csv")
econame = np.array(ecodata.name)
ecoresname = np.array(ecodata.resname)
ecoradeg = np.array(ecodata.radeg)
ecodedeg = np.array(ecodata.dedeg)
ecocz = np.array(ecodata.cz)
ecoabsrmag = np.array(ecodata.absrmag)
ecogiantsel = (ecodata.absrmag<-19.4)
ecogiantfofid = fof.fast_fof(ecoradeg[ecogiantsel],ecodedeg[ecogiantsel],ecocz[ecogiantsel],0.07,1.1,3.39/0.7,H0=70.)

FoF complete in 25.7107 s


In [4]:
ecogiantgrpra, ecogiantgrpdec, ecogiantgrpcz = fof.group_skycoords(ecoradeg[ecogiantsel], ecodedeg[ecogiantsel], ecocz[ecogiantsel], ecogiantfofid)
relvel = np.abs(ecogiantgrpcz - ecocz[ecogiantsel])
relprojdist = (ecogiantgrpcz + ecocz[ecogiantsel])/HUBBLE_CONST * ic.angular_separation(ecogiantgrpra, ecogiantgrpdec, ecoradeg[ecogiantsel], ecodedeg[ecogiantsel])/2.0
ecogiantgrpn = fof.multiplicity_function(ecogiantfofid, return_by_galaxy=True)
uniqecogiantgrpn, uniqindex = np.unique(ecogiantgrpn, return_index=True)
keepcalsel = np.where(uniqecogiantgrpn>1)

median_relprojdist = np.array([np.median(relprojdist[np.where(ecogiantgrpn==sz)]) for sz in uniqecogiantgrpn[keepcalsel]])
median_relvel = np.array([np.median(relvel[np.where(ecogiantgrpn==sz)]) for sz in uniqecogiantgrpn[keepcalsel]])

rproj_median_error = np.std(np.array([sbs(relprojdist[np.where(ecogiantgrpn==sz)], 10000, np.median, kwargs=dict({'axis':1 })) for sz in uniqecogiantgrpn[keepcalsel]]), axis=1)
dvproj_median_error = np.std(np.array([sbs(relvel[np.where(ecogiantgrpn==sz)], 10000, np.median, kwargs=dict({'axis':1})) for sz in uniqecogiantgrpn[keepcalsel]]), axis=1)

In [7]:
poptrproj,cov1 = curve_fit(giantmodel, uniqecogiantgrpn[keepcalsel], median_relprojdist, sigma=rproj_median_error)#, p0=[0.1, -2, 3, -0.1])
poptdvproj,cov2 = curve_fit(giantmodel, uniqecogiantgrpn[keepcalsel], median_relvel, sigma=dvproj_median_error)#, p0=[160,6.5,45,-600]) 
print("Giant model params.", poptrproj, poptdvproj)
print("errors: ",np.sqrt(np.diag(cov1)),np.sqrt(np.diag(cov2)))
rproj_boundary = lambda N: 3.5*giantmodel(N, *poptrproj) #3*(rprojslope*N+rprojint)
vproj_boundary = lambda N: 6.5*giantmodel(N, *poptdvproj) #4.5*(dvprojslope*N+dvprojint)
assert ((rproj_boundary(1)>0) and (vproj_boundary(1)>0)), "Cannot extrapolate Rproj or Vproj to N=1"

# get virial radii from abundance matching to giant-only groups
gihaloid, gilogmh, gir337b, gihalovdisp = ic.HAMwrapper(ecoradeg[ecogiantsel], ecodedeg[ecogiantsel], ecocz[ecogiantsel], ecoabsrmag[ecogiantsel], ecogiantfofid,\
                                                            192351., inputfilename=None, outputfilename=None)
gihalorvir = (3*(10**gilogmh) / (4*np.pi*337*0.3*2.77e11) )**(1/3.)
gihalon = np.array(fof.multiplicity_function(np.sort(ecogiantfofid), return_by_galaxy=False))
cvir= 11.*(10**gilogmh/4e12)**(-0.13)
Ac = np.log(1+cvir)-cvir/(1+cvir)
GRAV=4.32e-9
Vmax2 = (GRAV*(10**gilogmh)/gihalorvir)*0.216*cvir/Ac
gihalorvir = gihalorvir / (HUBBLE_CONST/100.)

Giant model params. [0.39380537 0.27639026] [ 3.48732769e+02 -1.63921838e-01]
errors:  [0.05546564 0.05911486] [3.65184085e+01 2.32298232e-02]


In [9]:
fig, (ax,ax1) = plt.subplots(ncols=2, figsize=doublecolsize)
sel = (ecogiantgrpn>1)
ax1.plot(gihalon-0.2, gihalovdisp, 'D', label=r'ECO HAM Velocity Dispersion', rasterized=True, ms=2, markerfacecolor="None", markeredgecolor='skyblue')
ax1.plot(ecogiantgrpn[sel]+0.1, relvel[sel], 'r.', alpha=0.2, label=r'ECO Giant Galaxies', rasterized=True)
ax1.errorbar(uniqecogiantgrpn[keepcalsel], median_relvel, fmt='k^', label=r'$\Delta v_{\rm proj}$ (Median of $\Delta v_{\rm proj,\, gal}$)',yerr=dvproj_median_error, rasterized=True, zorder=15)
tx = np.linspace(1,max(ecogiantgrpn),1000)
ax1.plot(tx, giantmodel(tx, *poptdvproj), label=r'$1\Delta v_{\rm proj}^{\rm fit}$', rasterized=True, color='blue')
ax1.plot(tx, 6.5*giantmodel(tx, *poptdvproj), 'g',  label=r'$6.5\Delta v_{\rm proj}^{\rm fit}$', linestyle='-.', rasterized=True)

ax1.set_xlim(0,20)
ax1.set_ylim(0,850)
ax1.set_xticks(np.arange(0,22,2))
ax1.set_xlabel("Number of Giant Members in Galaxy Group")
ax1.set_ylabel(r"Relative Velocity from Giant to Group Center [km s$^{-1}$]")
ax1.legend(loc='upper right', framealpha=1)

ax.plot(gihalon-0.1, gihalorvir, 'D', markeredgecolor='skyblue', markerfacecolor="None", ms=2, label=r'ECO HAM Virial Radii', rasterized=True)
ax.plot(ecogiantgrpn[sel]+0.2, relprojdist[sel], 'r.', alpha=0.2, label=r'ECO Giant Galaxies', rasterized=True)
ax.errorbar(uniqecogiantgrpn[keepcalsel], median_relprojdist, fmt='k^', label=r'$R_{\rm proj}$ (Median of $R_{\rm proj,\, gal}$)',yerr=rproj_median_error, rasterized=True, zorder=15)
ax.plot(tx, giantmodel(tx, *poptrproj), label=r'$1R_{\rm proj}^{\rm fit}$', rasterized=True, color='blue')
ax.plot(tx, 3.5*giantmodel(tx, *poptrproj), 'g', label=r'$3.5R_{\rm proj}^{\rm fit}$', linestyle='-.', rasterized=True)
ax.set_xlabel("Number of Giant Members in Galaxy Group")
ax.set_ylabel(r"Projected Distance from Giant to Group Center [Mpc]")
ax.legend(loc='best', framealpha=1)
ax.set_xlim(0,20)
ax.set_ylim(0,0.85)
ax.set_xticks(np.arange(0,22,2))
plt.tight_layout()
plt.savefig("../figures/rproj_calibration_assoc.pdf")
plt.show()

# Giant-only Group Multiplicity Function

In [10]:
eco = pd.read_csv("ECOdata_G3catalog_luminosity.csv")
giantsel = (eco.absrmag<-19.5)
plt.figure(figsize=(singlecolsize[0],1.1*singlecolsize[1]))
binv = np.arange(0.5,100.5,1)

ecogiantsel = (eco.absrmag<-19.4)
plt.hist(fof.multiplicity_function(eco.g3grp_l[ecogiantsel], return_by_galaxy=False), bins=binv, histtype='step', linewidth=3, label='ECO', color='palegreen')
#plt.hist(fof.multiplicity_function(resbg3grp[resbg3grp!=-99.], return_by_galaxy=False), bins=binv, histtype='step', linewidth=1.5, hatch='\\', label='RESOLVE-B', color='k')
plt.xlabel("Number of Giant Galaxies per Group")
plt.ylabel("Number of Giant-Only Groups")
plt.yscale('log')
plt.legend(loc='best')
plt.xlim(0.5,30)
plt.tight_layout()
plt.savefig("../figures/giantonlymult.pdf")
plt.show()

# Boundaries for Dwarf-Only Group Finding

In [13]:
ecodata = pd.read_csv("ECOdata_G3catalog_luminosity.csv")
ecoradeg = ecodata['radeg'].to_numpy()
ecodedeg = ecodata['dedeg'].to_numpy()
ecocz = ecodata['cz'].to_numpy()
ecoabsrmag = ecodata['absrmag'].to_numpy()
ecog3grp = ecodata['g3grp_l'].to_numpy()

In [19]:
ecogdgrpn = fof.multiplicity_function(ecog3grp, return_by_galaxy=True)
ecogdsel = np.logical_not(np.logical_or(ecog3grp==-99., ((ecogdgrpn==1) & (ecoabsrmag>-19.4) & (ecoabsrmag<=-17.33)))) #-17.33 not -17.0 (2/22/21)
ecogdgrpra, ecogdgrpdec, ecogdgrpcz = fof.group_skycoords(ecoradeg[ecogdsel], ecodedeg[ecogdsel], ecocz[ecogdsel], ecog3grp[ecogdsel])
ecogdrelvel = np.abs(ecogdgrpcz - ecocz[ecogdsel])
ecogdrelprojdist = (ecogdgrpcz + ecocz[ecogdsel])/HUBBLE_CONST * np.sin(ic.angular_separation(ecogdgrpra, ecogdgrpdec, ecoradeg[ecogdsel], ecodedeg[ecogdsel])/2.0)
ecogdn = ecogdgrpn[ecogdsel]
ecogdtotalmag = ic.get_int_mag(ecoabsrmag[ecogdsel], ecog3grp[ecogdsel])

magbins=np.arange(-24,-19,0.25)
binsel = np.where(np.logical_and(ecogdn>1, ecogdtotalmag>-24)) # test here
gdmedianrproj, magbincenters, agbinedges, jk = center_binned_stats(ecogdtotalmag[binsel], ecogdrelprojdist[binsel], np.median, bins=magbins)
gdmedianrproj_err, jk, jk, jk = center_binned_stats(ecogdtotalmag[binsel], ecogdrelprojdist[binsel], sigmarange, bins=magbins)
gdmedianrelvel, jk, jk, jk = center_binned_stats(ecogdtotalmag[binsel], ecogdrelvel[binsel], np.median, bins=magbins)
gdmedianrelvel_err, jk, jk, jk = center_binned_stats(ecogdtotalmag[binsel], ecogdrelvel[binsel], sigmarange, bins=magbins)
nansel = np.isnan(gdmedianrproj)

In [20]:
if 0: 
    guess=None
else:
    guess=[1e-5, 0.4]
poptr, pcovr = curve_fit(decayexp, magbincenters[~nansel], gdmedianrproj[~nansel], p0=guess)
print("guess:", poptr)
poptv, pcovv = curve_fit(decayexp, magbincenters[~nansel], gdmedianrelvel[~nansel], p0=[3e-5,4e-1])#,1])
print('dwarf only model params: ', poptr, poptv)
print('errors: ', np.sqrt(np.abs(np.diag(pcovr))), np.sqrt(np.abs(np.diag(pcovv))))

guess: [2.00553754e-06 4.90547314e-01]
dwarf only model params:  [2.00553754e-06 4.90547314e-01] [0.00155156 0.48162762]
errors:  [1.06548237e-06 2.30599254e-02] [0.00098522 0.02757992]


In [25]:
tx = np.linspace(-27,-17,100)
fig, (ax,ax1) = plt.subplots(ncols=2, figsize=doublecolsize)

giantgrpn = np.array([np.sum((ecoabsrmag[ecogdsel][ecog3grp[ecogdsel]==gg]<-19.4)) for gg in ecog3grp[ecogdsel]])
sel_ = np.where(np.logical_and(giantgrpn==1,ecogdtotalmag>-24))
ax.plot(ecogdtotalmag[sel_], ecogdrelprojdist[sel_], '.', color='mediumorchid', alpha=0.6, label=r'ECO $N_{\rm giants}=1$ Group Galaxies', rasterized=True)
sel_ = np.where(np.logical_and(giantgrpn==2,ecogdtotalmag>-24))
ax.plot(ecogdtotalmag[sel_], ecogdrelprojdist[sel_], '.', color='lawngreen', alpha=0.6, label=r'ECO $N_{\rm giants}=2$ Group Galaxies', rasterized=True)
sel_ = np.where(np.logical_and(giantgrpn>2,ecogdtotalmag>-24))
ax.plot(ecogdtotalmag[sel_], ecogdrelprojdist[sel_], '.', color='slategrey', alpha=0.6, label=r'ECO $N_{\rm giants}\geq3$ Group Galaxies', rasterized=True)
#ax.errorbar(magbincenters, gdmedianrproj, yerr=gdmedianrproj_err, fmt='k^', label=r'Medians ($R_{\rm proj}^{\rm gi,\,dw}$)', rasterized=True, zorder=15)
ax.errorbar(magbincenters, gdmedianrproj, yerr=None, fmt='k^', label=r'Medians ($R_{\rm proj}^{\rm gi,\,dw}$)', rasterized=True, zorder=15)
ax.plot(tx, 1*decayexp(tx,*poptr), label=r'$1R_{\rm proj,\,fit}^{\rm gi,\, dw}$', rasterized=True)
ax.plot(tx, 1*decayexp(tx,*poptr), label=r'$1R_{\rm proj,\,fit}^{\rm gi,\, dw}$', rasterized=True,linestyle='--')
#ax.plot(tx, 3*decayexp(tx,*poptr), label=r'$3R_{\rm proj,\,fit}^{\rm gi,\, dw}$', rasterized=True)
ax.set_xlabel(r"Integrated $M_r$ of Giant + Dwarf Members")
ax.set_ylabel(r"Projected Distance from Galaxy to Group Center [Mpc]")
ax.legend(loc='best',fontsize=8,framealpha=1)
ax.set_xlim(-24.1,-19)
ax.set_ylim(0,0.8)
ax.invert_xaxis()

#ax1.plot(ecogdtotalmag[binsel], ecogdrelvel[binsel], '.', alpha=0.6, label='ECO Giant-Hosting Group Galaxies', rasterized=True, color='palegreen')
#ax1.errorbar(magbincenters, gdmedianrelvel, yerr=gdmedianrelvel_err, fmt='k^',label=r'Medians ($\Delta v_{\rm proj}^{\rm gi,\,dw}$)', rasterized=True, zorder=15)
ax1.errorbar(magbincenters, gdmedianrelvel, yerr=None, fmt='k^',label=r'Medians ($\Delta v_{\rm proj}^{\rm gi,\,dw}$)', rasterized=True, zorder=15)
sel_ = np.where(np.logical_and(giantgrpn==1,ecogdtotalmag>-24))
ax1.plot(ecogdtotalmag[sel_], ecogdrelvel[sel_], '.', color='mediumorchid', alpha=0.6, label=r'ECO $N_{\rm giants}=1$ Group Galaxies', rasterized=True)
sel_ = np.where(np.logical_and(giantgrpn==2,ecogdtotalmag>-24))
ax1.plot(ecogdtotalmag[sel_], ecogdrelvel[sel_], '.', color='lawngreen', alpha=0.6, label=r'ECO $N_{\rm giants}=2$ Group Galaxies', rasterized=True)
sel_ = np.where(np.logical_and(giantgrpn>2,ecogdtotalmag>-24))
ax1.plot(ecogdtotalmag[sel_], ecogdrelvel[sel_], '.', color='slategrey', alpha=0.6, label=r'ECO $N_{\rm giants}\geq3$ Group Galaxies', rasterized=True)
ax1.plot(tx, decayexp(tx, *poptv), label=r'$\Delta v_{\rm proj,\, fit}^{\rm gi,\, dw}$', rasterized=True)
ax1.plot(tx, 2.5*decayexp(tx, *poptv), label=r'$2.5\Delta v_{\rm proj,\, fit}^{\rm gi,\, dw}$', rasterized=True, linestyle='--')
ax1.set_ylabel(r"Relative Velocity from Galaxy to Group Center [km s$^{-1}]$")
ax1.set_xlabel(r"Integrated $M_r$ of Giant + Dwarf Members")
print("Fraction outside of iterative combination boundaries: ")
print(np.sum(ecogdrelprojdist[binsel]>3*decayexp(ecogdtotalmag[binsel],*poptr))/len(ecogdtotalmag[binsel]))
print(np.sum(ecogdrelvel[binsel]>4.5*decayexp(ecogdtotalmag[binsel],*poptv))/len(ecogdtotalmag[binsel]))
ax1.set_xlim(-24.1,-19)
ax1.set_ylim(0,800)
ax1.invert_xaxis()
ax1.legend(loc='best',fontsize=8, framealpha=1)
plt.tight_layout()
plt.savefig("../figures/itercombboundaries.pdf")
plt.show()

Fraction outside of iterative combination boundaries: 
0.033543577981651376
0.01576834862385321


# $M_{\rm halo} - L_{\rm tot}$ Curve

In [27]:
ecodata = pd.read_csv("ECOdata_G3catalog_luminosity.csv")
sel = (ecodata.g3logmh_l>0)
ecologmh = ecodata.g3logmh_l[sel]
ecoLtot = ic.get_int_mag(ecodata[sel].absrmag.to_numpy(), ecodata[sel].g3grp_l.to_numpy())

In [ ]:
# resb stuff

In [28]:
plt.figure(figsize=singlecolsize)
plt.plot(ecoLtot, ecologmh, '.', color='palegreen', alpha=0.6, label='ECO', markersize=11, rasterized=True)
plt.xlabel("group-integrated r-band luminosity")
plt.ylabel(r"group halo mass (log$M_\odot$)")
plt.legend(loc='best')
plt.gca().invert_xaxis()
plt.tight_layout()
plt.savefig("../figures/hamLrrelationG3.pdf")
plt.show()

# G3 vs. FoF Multiplicity Functions

In [29]:
ecodata = pd.read_csv("ECOdata_G3catalog_luminosity.csv")
ecodata = ecodata[ecodata.g3grp_l>0]
ecoradeg = ecodata['radeg'].to_numpy()
ecodedeg = ecodata['dedeg'].to_numpy()
ecocz = ecodata['cz'].to_numpy()
ecoabsrmag = ecodata['absrmag'].to_numpy()
ecog3grp = ecodata['g3grp_l'].to_numpy()

In [30]:
binv = np.arange(0.5,1200.5,1)
g3mult=np.array(fof.multiplicity_function(ecog3grp[ecog3grp!=-99.], return_by_galaxy=False))
g3hist, binedges = np.histogram(g3mult,bins=binv)
bincenters=0.5*(binedges[1:]+binedges[:-1])
sel=(g3hist>0)
g3popt, g3pcov = curve_fit(decayexp, bincenters[sel],np.log10(g3hist[sel]))

fig, (ax,ax2) = plt.subplots(figsize=(doublecolsize[0],0.6*doublecolsize[1]), ncols=2, sharey=True)
binv = np.arange(0.5,1200.5,1)
ax.hist(fof.multiplicity_function(ecog3grp[ecog3grp!=-99.], return_by_galaxy=False), bins=binv, log=True, label='ECO (All)', histtype='step', linewidth=3, color='green')
#ax.hist(fof.multiplicity_function(resbg3grp[resbg3grp!=-99.], return_by_galaxy=False), bins=binv, log=True, label='RESOLVE-B (All)', histtype='step', color='k')
ax.set_xlabel("Number of Group Members")
ax.annotate("G3",xy=(20,10),fontsize=14)
ax.set_ylabel("Number of Groups")
ax.set_xlim(0.5,30)

binvd=binv
ax.hist(fof.multiplicity_function(ecoitassocid, return_by_galaxy=False), bins=binvd, log=True, histtype='stepfilled', color='palegreen', label='ECO (Dwarf-Only)')
ax.hist(fof.multiplicity_function(resbitassocid, return_by_galaxy=False), bins=binvd, log=True, histtype='step', alpha=0.9, color='k', hatch='//', label='RESOLVE-B (Dwarf-Only)')
ax.plot(binv,10**decayexp(binv,*g3popt),color='purple',label='Fit to ECO G3 Groups')
ax.legend(loc='best')


ecodr2=pd.read_csv("ECODR2.csv")
ecodr2=ecodr2[ecodr2.absrmag<-17.33] 
ax2.hist(fof.multiplicity_function(ecodr2.grp_e17[ecodr2.grp_e17>0], return_by_galaxy=False), bins=binv, log=True, label='ECO (All)', histtype='step', linewidth=3, color='green')
ax2.hist(fof.multiplicity_function(np.array(resolvedata.grp_e17[(resolvedata.f_b==1)&(resolvedata.grp_e17!=-99.)]), return_by_galaxy=False), bins=binv, log=True, label='RESOLVE-B (All)', histtype='step', color='k')
fofdwarfonly = ecodr2[(ecodr2.grp_e17!=-99.)].groupby('grp_e17').filter(lambda grp_e17:(grp_e17.absrmag>-19.4).all())
ax2.hist(fof.multiplicity_function(np.array(fofdwarfonly.grp_e17), return_by_galaxy=False), bins=binvd, log=True, histtype='stepfilled', color='palegreen', label='ECO (Dwarf-Only)')
fofdwarfonly = resolvedata[(resolvedata.grp_e17!=-99.)&(resolvedata.f_b==1)].groupby('grp_e17').filter(lambda grp_e17:(grp_e17.absrmag>-19.4).all())
ax2.hist(fof.multiplicity_function(np.array(fofdwarfonly.grp_e17), return_by_galaxy=False), bins=binvd, log=True, histtype='step', color='k', alpha=0.9, hatch='//', label='RESOLVE-B (Dwarf-Only)')
ax2.plot(binv,10**decayexp(binv,*g3popt),color='purple',label='Fit to ECO G3 Groups')
ax2.annotate("FoF (E17)",xy=(15,10),fontsize=14)
ax2.set_xlim(0.5,30)
ax2.set_xlabel("Number of Group Members")
ax2.legend(loc='best')
plt.tight_layout()
plt.savefig("paper1plots/multfunc_doinset.pdf")
plt.show()

print("K-S test for G3 vs. FOF E17 Mult. Functions (ECO):")
fofmult=np.array(fof.multiplicity_function(ecodr2.grp_e17[ecodr2.grp_e17>0], return_by_galaxy=False))
g3mult=np.array(fof.multiplicity_function(ecog3grp[ecog3grp!=-99.], return_by_galaxy=False))
print(kstest(g3mult[g3mult>1],fofmult[fofmult>1]))
print(kstest(g3mult[g3mult>1],fofmult[fofmult>1],'less'))
print(kstest(g3mult[g3mult>1],fofmult[fofmult>1],'greater'))
print("hellinger distance: ")
g3hist,_ = np.histogram(g3mult,bins=binv,density=True)
fofhist,_ = np.histogram(fofmult,bins=binv,density=True)
print(hellinger2(g3hist,fofhist))

NameError: name 'ecoitassocid' is not defined